# 第 16 章：时间序列 I

本 Notebook 用固定种子构造两条具有相同理论均值和方差、但不同序列相关的月度序列，再以逐月扩展训练窗比较 ARIMA(1,0,0) 与训练均值基准。它也读取项目已保存的 French 10 行业月度组合收益，用于当前发布快照的描述统计。

模拟序列只验证算法和已知生成机制；French 历史组合是当前版本快照，缺少逐月历史可用时间，不用于历史样本外交易预测。所有核心验证、估计和滚动预测函数来自 `src/quant_course/time_series.py`。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch16_time_series_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
from base64 import b64encode
from IPython.display import HTML, display

from quant_course.time_series import (
    ar1_theoretical_moments,
    fit_arima,
    forecast_arima,
    ljung_box_diagnostic,
    rolling_origin_forecast,
    sample_acf_pacf,
    simulate_stationary_pair,
    split_monthly_series,
    training_mean_forecast,
    validate_monthly_series,
)

print("project root:", ROOT)

## 两条构造序列：均值、方差相同，自相关不同

AR(1) 创新标准差取 `marginal_sd * sqrt(1 - phi**2)`，因此理论无条件标准差与 IID 正态序列相同。有限样本的样本标准差不会被强制设成一样。

In [ ]:
pair = simulate_stationary_pair(n=480, seed=20261016, mean=0.005, marginal_sd=0.04, phi=0.7, start="1980-01")
moments = ar1_theoretical_moments(mean=0.005, phi=0.7, innovation_sd=0.04 * np.sqrt(1 - 0.7**2), max_lag=3)
print("AR(1) theoretical mean / variance / ACF:", moments.mean, moments.variance, moments.autocorrelations)
for column in ("iid", "ar1"):
    summary = sample_acf_pacf(pair[column], lags=12)
    print(column, "sample mean", pair[column].mean(), "sample sd", pair[column].std(ddof=1), "ACF(1)", summary.loc[1, "acf"])

figure_path = ROOT / "fixtures/figures/a4/ch16_stationarity_acf.png"
assert figure_path.is_file(), f"run code/ch16_time_series.py first to generate {figure_path}"
figure_data = b64encode(figure_path.read_bytes()).decode("ascii")
figure_alt = "四格图比较月度收益百分比路径和按月滞后的自相关：Gaussian IID 的理论 ACF 为零，phi 为 0.70 的平稳 AR(1) 理论 ACF 按 0.70 的幂衰减。"
display(HTML(f'<img alt="{figure_alt}" src="data:image/png;base64,{figure_data}" style="max-width: 100%; height: auto;">'))

## 只在历史前缀拟合，并滚动预测下一个月

下列切分使用前 360 个月训练。每次滚动起点只把当时之前的月份交给拟合函数，目标是紧邻的下一月。Ljung–Box 检验检查训练残差截至 12 个月的联合线性自相关；高 p 值不能证明模型正确，也不检查波动聚集。

In [ ]:
train, test = split_monthly_series(pair["ar1"], train_size=360)
forecast = forecast_arima(train, steps=3, order=(1, 0, 0), alpha=0.05)
fitted = fit_arima(train, order=(1, 0, 0))
ljung_box = ljung_box_diagnostic(fitted.residuals, model_df=1, lag=12)
rolling = rolling_origin_forecast(pair["ar1"], initial_train_size=360, order=(1, 0, 0))
rolling_mean = np.array([pair["ar1"].iloc[:i].mean() for i in range(360, len(pair))])
rmse_ar = np.sqrt(np.mean((rolling["actual"].to_numpy() - rolling["forecast"].to_numpy()) ** 2))
rmse_mean = np.sqrt(np.mean((rolling["actual"].to_numpy() - rolling_mean) ** 2))
print("training through", train.index[-1], "test begins", test.index[0])
print(forecast)
print("Ljung-Box residual check at lag 12:")
print(ljung_box)
print("simulated expanding-window RMSE ARIMA / train mean:", rmse_ar, rmse_mean)
print("first target and its information origin:", rolling.iloc[0]["origin_period"], "->", rolling.index[0])

## French 行业组合：只作当前快照描述

处理后的 CSV 已选取官方原始文件的月度价值加权行业收益块，并将百分数转为小数收益。`month` 是 `YYYYMM`，故读取时保留字符串再构造成月度 `PeriodIndex`。快照只足以做历史描述统计；发布版本没有逐月 `available_at`，不能证明某行当时可用，更不能据此声称真实样本外预测能力。

In [ ]:
if RUN_FRENCH:
    french_path = ROOT / "data/processed/french_10_industries_monthly.csv"
    french_table = pd.read_csv(french_path, dtype={"month": "string"})
    french_index = pd.PeriodIndex(french_table.pop("month"), freq="M", name="month")
    no_dur = validate_monthly_series(pd.Series(french_table["NoDur"].to_numpy(dtype=float), index=french_index, name="NoDur"), min_observations=24)
    french_acf = sample_acf_pacf(no_dur, lags=12)
    print("French current snapshot, decimal monthly returns:", no_dur.index[0], "to", no_dur.index[-1], "n=", len(no_dur))
    print("NoDur mean / standard deviation:", no_dur.mean(), no_dur.std(ddof=1))
    print("descriptive ACF(1), ACF(12):", french_acf.loc[1, "acf"], french_acf.loc[12, "acf"])
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 解释边界

模型阶数、窗口、参数估计、预测区间和基准都属于研究设计。真实交易还需要核对每项数据的可得时间、信号何时形成、订单何时能执行、目标收益覆盖哪个区间，并扣除实际成本。本 Notebook 未定义 French 数据的历史发布时间或成交规则；也未执行单位根、ADF、VAR 或协整检验，这些主题留在第 17 章。